In [7]:
import arcpy
import os
import csv
import math

# =====================================================
# USER INPUTS
# =====================================================
ROW_FC = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\ROW_Width_Null_Polygons\Null_Width_Polygons.gdb\CONUS_Null_Width_Polygons"
OUT_GDB = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\ROW_Width_Null_Polygons\Calculated_ROWs_MRR.gdb"
OUT_PREFIX = "CONUS_ROW_MRR"

ROW_ID_FIELD = "ROW_ID_OG"
WIDTH_FIELD = "approx_length_meters"
CSV_OUTPUT = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\ROW_Width_Null_Polygons\CONUS_row_widths_MRR.csv"

arcpy.env.overwriteOutput = True
arcpy.env.workspace = OUT_GDB

# =====================================================
# STEP 1: MINIMUM ROTATED RECTANGLES
# =====================================================
mrr_fc = os.path.join(OUT_GDB, f"{OUT_PREFIX}_MBR")

arcpy.management.MinimumBoundingGeometry(
    ROW_FC,
    mrr_fc,
    "RECTANGLE_BY_AREA"
)

# =====================================================
# STEP 2: COMPUTE WIDTH FROM RECTANGLE GEOMETRY
# =====================================================
width_dict = {}

with arcpy.da.SearchCursor(mrr_fc, [ROW_ID_FIELD, "SHAPE@"]) as cursor:
    for row_id, geom in cursor:
        if geom is None:
            continue

        # Rectangle has one part with 5 points (last = first)
        pts = geom.getPart(0)

        edges = []
        for i in range(4):
            p1 = pts[i]
            p2 = pts[i + 1]
            length = math.hypot(p2.X - p1.X, p2.Y - p1.Y)
            edges.append(length)

        # Two unique edge lengths → width is the smaller
        width_dict[row_id] = min(edges)

# =====================================================
# STEP 3: WRITE WIDTH BACK TO ROW POLYGONS
# =====================================================
row_fields = [f.name for f in arcpy.ListFields(ROW_FC)]
if WIDTH_FIELD not in row_fields:
    arcpy.management.AddField(ROW_FC, WIDTH_FIELD, "DOUBLE")

with arcpy.da.UpdateCursor(ROW_FC, [ROW_ID_FIELD, WIDTH_FIELD]) as cursor:
    for row_id, _ in cursor:
        cursor.updateRow([row_id, width_dict.get(row_id)])

# =====================================================
# STEP 4: OPTIONAL CSV OUTPUT
# =====================================================
if CSV_OUTPUT:
    with open(CSV_OUTPUT, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([ROW_ID_FIELD, WIDTH_FIELD])
        for k, v in width_dict.items():
            writer.writerow([k, v])

print("ROW width calculation complete — geometry-verified.")


ROW width calculation complete — geometry-verified.


In [10]:
import arcpy
import csv

# ============================
# USER INPUTS
# ============================
ROW_FC = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\ROW_Width_Null_Polygons\Final_ROWs_Exclusion_200m_AllWidths.gdb\CONUS_Final_ROWs_Exclusion_200m_AllWidths"
ROW_ID_FIELD = "ROW_ID_OG"
WIDTH_FIELD = "approx_length_meters"

# CSV with calculated widths
CSV_FILE = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\ROW_Width_Null_Polygons\CONUS_row_widths_MRR.csv"
# CSV format: ROW_ID_OG, approx_length_meters
# e.g. 1, 35.2

# ============================
# LOAD CSV INTO DICTIONARY
# ============================
width_dict = {}
with open(CSV_FILE, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = row[ROW_ID_FIELD]
        try:
            val = float(row[WIDTH_FIELD])
            width_dict[key] = val
        except (ValueError, KeyError):
            # Skip rows with missing/invalid data
            continue

# ============================
# UPDATE ROW_FC ONLY WHERE NULL
# ============================
with arcpy.da.UpdateCursor(ROW_FC, [ROW_ID_FIELD, WIDTH_FIELD]) as cursor:
    for row_id, width in cursor:
        # Only update if currently NULL
        if width is None:
            new_width = width_dict.get(str(row_id))  # CSV keys are read as strings
            if new_width is not None:
                cursor.updateRow([row_id, new_width])

print("Null approx_length_meters values updated successfully.")


Null approx_length_meters values updated successfully.
